# Estudio 06 · Fase 5 · Conteo

Preregistro `PHASE5` (commit `6d2164c`, aprobado por Rosa con una lectura agregada). Base: L. Decide la razón MAE
(4 de 5 temporadas a favor y ninguna en contra); vetan razón RMSE, F1, AUPRC, Brier (3 de 5) y la antelación si
empeora. Se adopta con «gana» en ≥ 2 de 3 horizontes y sin «pierde» ni «compensa».

- **5a:** mezcla con la persistencia, w por horizonte y bloque con las OOS de los bloques anteriores (solo el conteo).
- **5b:** `max_delta_step` elegido por bloque en {0,7; 1; 2; 3; 5; 10} con validación interna.
- **5c:** 3a2 (peso por temporada [0,25, 1]), predicciones guardadas de la fase 3; no es ciega.


In [1]:
import json
import sys

import numpy as np
import pandas as pd

from denguekay.config.paths import RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.evaluation import protocol as pa
from denguekay.model import tracking
from denguekay.model.device import xgboost_settings

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase1_history as p1
import phase2_calibration as p2
import phase3_onsets as p3
import phase5_counts as p5
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "fase5.json"
device = xgboost_settings()
inp = p3.phase3_inputs()
device


{'device': 'cuda', 'n_jobs': 12}

## 1 · Guardas G12 y G13 (G11 se comprueba tras la mezcla)


In [2]:
saved = p2.read_l_predictions()
columns, hp = p3._config()
ref17 = p3.predict_blocks_weighted(inp["matrices"], inp["origenes"], p1.seasonal_blocks(p1.FIRST_BLOCK_SEASON_LONG),
                                   17, columns, hp, hyperparameters_fn=p5.mds_selector(hp, grid=(0.7,)))
ref17.attrs.pop("ajustes")
g12 = p1.compare_predictions(ref17, saved[17])
s3a2 = p5.read_3a2()
key = ["ubigeo", "semana_inicio", "horizonte", "bloque"]
g13 = {"semillas": sorted(s3a2) == list(pr.SEEDS),
       "mismas_filas": all(s3a2[s][key].astype(str).equals(saved[s][key].astype(str)) for s in pr.SEEDS)}
g13["pasa"] = all(g13.values())
if not (g12["pasa"] and g13["pasa"]):
    raise RuntimeError(f"Falla una guarda: se detiene la fase. G12={g12} G13={g13}")
{"G12": g12, "G13": g13}


{'G12': {'filas': 183105,
  'filas_guardadas': 183105,
  'cruzadas': 183105,
  'diferencia_maxima': {'casos_predichos': 0.0, 'probabilidad_brote': 0.0},
  'pasa': True},
 'G13': {'semillas': True, 'mismas_filas': True, 'pasa': True}}

## 2 · Variantes (5 semillas)


In [3]:
L = p3.reference_l(inp["serie"])
A = {"evaluaciones": {s: p5.evaluate_mixture(p, inp["serie"]) for s, p in saved.items()}}
w = pd.concat({s: e["w"] for s, e in A["evaluaciones"].items()}, names=["semilla"]).reset_index(level=0)
B = p5.run_mds(inp, pr.SEEDS)
Cc = {"evaluaciones": {s: pa.evaluate(p, inp["serie"]) for s, p in s3a2.items()}}
p5.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.concat(B["predicciones"].values(), ignore_index=True).to_csv(p5.OUTPUT_DIR / "predicciones_5b.csv.gz", index=False)
B["ajustes"].drop(columns=["corte"]).to_csv(p5.OUTPUT_DIR / "ajustes_5b.csv", index=False)
w.drop(columns=["corte"]).to_csv(p5.OUTPUT_DIR / "w_5a.csv", index=False)


In [4]:
# G11: en los bloques y horizontes con w = 1, el conteo mezclado es exactamente el de L
bad = 0
for s in pr.SEEDS:
    d, l = A["evaluaciones"][s]["datos"], L["evaluaciones"][s]["datos"]
    for r in w[w.semilla.eq(s) & w.w.eq(1.0)].itertuples():
        m = d.horizonte.eq(r.horizonte) & d.bloque.eq(r.bloque)
        ml = l.horizonte.eq(r.horizonte) & l.bloque.eq(r.bloque)
        bad += int(not np.array_equal(d.loc[m, "casos_predichos"].to_numpy(), l.loc[ml, "casos_predichos"].to_numpy()))
g11 = {"bloques_con_w_1_distintos": bad, "pasa": bad == 0}
if not g11["pasa"]:
    raise RuntimeError(f"Falla G11: se detiene la fase. {g11}")
g11


{'bloques_con_w_1_distintos': 0, 'pasa': True}

## 3 · Veredictos preregistrados


In [5]:
variants = {"5a": A, "5b": B, "5c": Cc}
dec = {v: p5.compare(x, L) for v, x in variants.items()}
adopted = [v for v in dec if pr.adopt_phase5(dec[v]["veredicto"])]
print("Adoptadas:", adopted)
pd.concat({v: d["veredicto"] for v, d in dec.items()}, names=["variante"]).reset_index(level=0)


Adoptadas: ['5a']


,variante,horizonte,veredicto,vetos,vetos_contra_referencia,a_favor_razon_mae,a_favor_razon_rmse,a_favor_f1_050,a_favor_auprc,a_favor_brier,en_contra_razon_mae,en_contra_razon_rmse,en_contra_f1_050,en_contra_auprc,en_contra_brier,a_favor_antelacion,en_contra_antelacion
0,5a,2,gana,None,razon_rmse,5,4,0,0,0,0,0,0,0,0,0,0
1,5a,3,gana,None,razon_rmse,4,4,0,0,0,0,0,0,0,0,0,0
2,5a,4,gana,None,razon_rmse,4,4,0,0,0,0,0,0,0,0,0,0
0,5b,2,empate,None,None,1,1,1,0,1,0,0,0,0,1,2,0
1,5b,3,empate,None,None,1,1,0,1,1,0,0,0,0,1,2,0
2,5b,4,empate,None,None,1,1,2,1,2,0,0,0,0,0,2,0
0,5c,2,empate,NaN,"razon_rmse, brier",3,4,2,2,3,0,0,1,0,1,0,1
1,5c,3,compensa (empate),brier,"razon_rmse, auprc",4,4,1,3,2,0,0,0,2,3,0,1
2,5c,4,empate,brier,razon_rmse,3,4,0,2,2,0,0,2,1,3,0,1


In [6]:
pd.concat({v: d["antelacion"] for v, d in dec.items()}, names=["variante"]).round(3)


horizonte  episodios  diferencia  ic95_inferior  ic95_superior  diferencia_temporada_2010  diferencia_temporada_2015  diferencia_temporada_2016  diferencia_temporada_2022  diferencia_temporada_2023  \
variante                                                                                                                                                                                                            
5a       0          2        193       0.000          0.000          0.000                        0.0                      0.000                      0.000                      0.000                      0.000   
         1          3        193       0.000          0.000          0.000                        0.0                      0.000                      0.000                      0.000                      0.000   
         2          4        193       0.000          0.000          0.000                        0.0                      0.000                      0.000                      0.000                      0.000   
5b       0          2        193       0.047          0.021          0.073                        0.0                      0.000                      0.000                      0.105                      0.088   
         1          3        193       0.016         -0.005          0.041                        0.0                      0.000                     -0.029                      0.053                      0.035   
         2          4        193       0.031          0.010          0.057                        0.0                      0.000                      0.000                      0.053                      0.070   
5c       0          2        193      -0.026         -0.052          0.000                        0.0                     -0.029                      0.000                      0.000                     -0.070   
         1          3        193      -0.036         -0.067         -0.005                        0.0                      0.000                      0.000                     -0.026                     -0.105   
         2          4        193      -0.031         -0.062          0.000                        0.0                      0.000                      0.000                      0.000                     -0.105   

            episodios_temporada_2010  episodios_temporada_2015  episodios_temporada_2016  episodios_temporada_2022  episodios_temporada_2023  
variante                                                                                                                                      
5a       0                        30                        34                        34                        38                        57  
         1                        30                        34                        34                        38                        57  
         2                        30                        34                        34                        38                        57  
5b       0                        30                        34                        34                        38                        57  
         1                        30                        34                        34                        38                        57  
         2                        30                        34                        34                        38                        57  
5c       0                        30                        34                        34                        38                        57  
         1                        30                        34                        34                        38                        57  
         2                        30                        34                        34                        38                        57

In [7]:
pd.concat({v: d["metricas"] for v, d in dec.items()}, names=["variante"]).round(3)


horizonte          bloque  positivos_cand  razon_mae_cand  razon_mae_ref  razon_rmse_cand  razon_rmse_ref  mae_cand  mae_ref  rmse_cand  rmse_ref  f1_050_cand  f1_050_ref  auprc_cand  auprc_ref  \
variante                                                                                                                                                                                                         
5a       3           2  temporada_2010           234.0           0.907          0.938            0.899           0.920     1.513    1.564     11.475    11.735        0.722       0.722       0.780      0.780   
         8           2  temporada_2015           557.0           0.889          0.945            0.932           1.053     2.644    2.811     11.334    12.808        0.772       0.772       0.885      0.885   
         9           2  temporada_2016           395.0           1.030          1.100            0.995           1.071     1.551    1.657      4.771     5.139        0.657       0.657       0.757      0.757   
         15          2  temporada_2022           431.0           0.946          0.962            0.965           1.010     1.853    1.884      5.823     6.097        0.712       0.712       0.781      0.781   
         16          2  temporada_2023          1039.0           0.808          0.847            0.893           0.940     8.752    9.177     44.156    46.507        0.857       0.857       0.947      0.947   
         17          2  temporada_2024           857.0           1.136          1.259            1.316           1.566     5.628    6.242     20.737    24.673        0.799       0.799       0.893      0.893   
         21          3  temporada_2010           234.0           0.891          0.897            0.890           0.891     1.972    1.985     14.690    14.703        0.631       0.631       0.706      0.706   
         26          3  temporada_2015           557.0           0.828          0.895            0.914           1.045     3.112    3.361     13.596    15.548        0.649       0.649       0.839      0.839   
         27          3  temporada_2016           395.0           1.042          1.131            1.016           1.142     1.862    2.021      5.732     6.445        0.627       0.627       0.709      0.709   
         33          3  temporada_2022           431.0           0.927          0.959            0.990           1.063     2.213    2.289      7.257     7.794        0.704       0.704       0.771      0.771   
         34          3  temporada_2023          1039.0           0.763          0.791            0.852           0.888    11.174   11.581     57.146    59.519        0.835       0.835       0.930      0.930   
         35          3  temporada_2024           857.0           1.238          1.382            1.485           1.778     7.895    8.816     30.661    36.696        0.788       0.788       0.876      0.876   
         39          4  temporada_2010           234.0           0.876          0.876            0.859           0.859     2.307    2.307     16.421    16.421        0.476       0.476       0.600      0.600   
         44          4  temporada_2015           557.0           0.793          0.847            0.880           0.993     3.466    3.702     15.395    17.369        0.480       0.480       0.811      0.811   
         45          4  temporada_2016           395.0           1.062          1.150            1.033           1.176     2.219    2.402      6.779     7.714        0.590       0.590       0.660      0.660   
         51          4  temporada_2022           431.0           0.929          0.952            1.038           1.114     2.644    2.709      8.864     9.505        0.652       0.652       0.743      0.743   
         52          4  temporada_2023          1039.0           0.733          0.748            0.815           0.839    12.960   13.217     65.313    67.236        0.811       0.811       0.922      0.922   
         53          4 

In [8]:
crosses = []
if len(adopted) > 1:
    order = [v for v in pr.PHASE5_SIMPLICITY if v in adopted]
    best = order[0]
    for other in order[1:]:
        c = p5.compare(variants[other], variants[best])
        wins = pr.adopt_phase5(c["veredicto"])
        crosses.append({"candidata": other, "frente_a": best, "adopta": wins,
                        "veredicto": "; ".join(f"h{r.horizonte}: {r.veredicto}" for r in c["veredicto"].itertuples())})
        best = other if wins else best
    final = best
else:
    final = adopted[0] if adopted else "L"
print("Base tras la fase 5:", final, crosses)


Base tras la fase 5: 5a []


## 4 · Lecturas que no deciden


In [9]:
pd.concat({v: d["lectura_sin_2010"] for v, d in dec.items()}, names=["variante"]).reset_index(level=0)


,variante,horizonte,veredicto,vetos,vetos_contra_referencia,a_favor_razon_mae,a_favor_razon_rmse,a_favor_f1_050,a_favor_auprc,a_favor_brier,en_contra_razon_mae,en_contra_razon_rmse,en_contra_f1_050,en_contra_auprc,en_contra_brier,a_favor_antelacion,en_contra_antelacion
0,5a,2,gana,None,razon_rmse,4,4,0,0,0,0,0,0,0,0,0,0
1,5a,3,gana,None,razon_rmse,4,4,0,0,0,0,0,0,0,0,0,0
2,5a,4,gana,None,razon_rmse,4,4,0,0,0,0,0,0,0,0,0,0
0,5b,2,empate,None,None,1,1,1,0,1,0,0,0,0,1,2,0
1,5b,3,empate,None,None,1,1,0,1,1,0,0,0,0,1,2,0
2,5b,4,empate,None,None,1,1,2,1,2,0,0,0,0,0,2,0
0,5c,2,gana,None,"razon_rmse, brier",3,4,2,2,3,0,0,0,0,1,0,1
1,5c,3,gana,None,"razon_rmse, auprc",3,4,1,3,2,0,0,0,1,2,0,1
2,5c,4,empate,None,razon_rmse,2,4,0,2,2,0,0,2,1,2,0,1


In [10]:
w_summary = w.groupby(["horizonte", "bloque"]).w.agg(["median", "min", "max"]).reset_index()
mds = B["ajustes"].groupby("bloque").hp_max_delta_step.agg(lambda x: sorted(x.tolist())).reset_index()
display(w_summary.round(2))
mds


,horizonte,bloque,median,min,max
0,2,calendario_2025,0.63,0.61,0.65
1,2,temporada_2008,1.00,1.00,1.00
2,2,temporada_2009,1.00,0.97,1.00
3,2,temporada_2010,0.88,0.85,0.89
4,2,temporada_2011,0.66,0.65,0.68
5,2,temporada_2012,0.62,0.61,0.63
6,2,temporada_2013,0.62,0.61,0.63
7,2,temporada_2014,0.62,0.61,0.63
8,2,temporada_2015,0.59,0.58,0.59
9,2,temporada_2016,0.61,0.59,0.62


,bloque,hp_max_delta_step
0,calendario_2025,"[0.7, 0.7, 0.7, 0.7, 0.7]"
1,temporada_2008,"[0.7, 0.7, 0.7, 0.7, 0.7]"
2,temporada_2009,"[0.7, 0.7, 0.7, 0.7, 0.7]"
3,temporada_2010,"[0.7, 0.7, 0.7, 0.7, 0.7]"
4,temporada_2011,"[1.0, 1.0, 1.0, 1.0, 1.0]"
5,temporada_2012,"[0.7, 0.7, 0.7, 0.7, 0.7]"
6,temporada_2013,"[0.7, 0.7, 0.7, 0.7, 0.7]"
7,temporada_2014,"[0.7, 0.7, 0.7, 0.7, 0.7]"
8,temporada_2015,"[0.7, 0.7, 1.0, 1.0, 1.0]"
9,temporada_2016,"[1.0, 1.0, 1.0, 1.0, 1.0]"


In [11]:
report = [pr.block_name(s) for s in (*pr.DECIDING_SEASONS, pr.CONTRAST_SEASON)]
phases = pd.concat({k: p5.phases_of_curve(v) for k, v in {"L": L, **variants}.items()}, names=["variante"])
phases = phases[phases.bloque.isin(report)]
phases[["horizonte", "bloque", "fase", "n", "mae_modelo", "rmse_modelo", "razon_mae", "razon_rmse"]].round(3)


horizonte          bloque    fase       n  mae_modelo  rmse_modelo  razon_mae  razon_rmse
variante                                                                                               
L        12           2  temporada_2010   valle  2340.0       0.343        1.965      0.986       0.885
         13           2  temporada_2010  subida   390.0       2.595       13.721      0.938       0.928
         14           2  temporada_2010    pico   195.0       9.261       39.894      1.060       1.065
         15           2  temporada_2010  bajada   455.0       3.695       13.108      0.823       0.659
         32           2  temporada_2015   valle  1950.0       0.589        2.921      0.921       0.925
         33           2  temporada_2015  subida   390.0       2.905       11.271      1.036       0.964
         34           2  temporada_2015    pico   650.0       9.460       26.950      1.139       1.221
         35           2  temporada_2015  bajada   455.0       2.762        7.572      0.501       0.467
         36           2  temporada_2016   valle   780.0       0.552        1.314      1.092       0.844
         37           2  temporada_2016  subida  1235.0       1.749        5.478      1.256       1.354
         38           2  temporada_2016    pico   780.0       2.985        7.654      1.091       1.014
         39           2  temporada_2016  bajada   585.0       1.154        2.950      0.801       0.674
         60           2  temporada_2022   valle  1625.0       0.649        1.858      0.929       0.857
         61           2  temporada_2022  subida   715.0       2.358        8.048      1.047       1.146
         62           2  temporada_2022    pico   585.0       4.651       10.504      1.091       1.133
         63           2  temporada_2022  bajada   455.0       2.021        4.531      0.666       0.559
         64           2  temporada_2023   valle  1950.0       1.298        4.515      1.004       0.996
         65           2  temporada_2023  subida   455.0      13.855       37.503      1.063       1.053
         66           2  temporada_2023    pico   390.0      42.350      126.941      0.931       1.029
         67           2  temporada_2023  bajada   585.0       9.614       23.819      0.539       0.439
         68           2  temporada_2024   valle  1235.0       1.506        3.430      0.920       0.811
         69           2  temporada_2024  subida  1040.0       3.306        7.840      0.782       0.702
         70           2  temporada_2024    pico   585.0      19.162       53.399      1.741       1.919
         71           2  temporada_2024  bajada   520.0       8.818       24.854      1.177       1.171
         84           3  temporada_2010   valle  2340.0       0.392        2.284      0.878       0.736
         85           3  temporada_2010  subida   390.0       3.009       16.050      0.941       0.925
         86           3  temporada_2010    pico   195.0      11.868       51.090      1.052       1.053
         87           3  temporada_2010  bajada   455.0       5.033       15.615      0.766       0.587
         104          3  temporada_2015   valle  1950.0       0.673        2.985      0.846       0.786
         105          3  temporada_2015  subida   390.0       3.327       11.666      0.962       0.963
         106          3  temporada_2015    pico   650.0      11.578       33.409      1.162       1.285
         107          3  temporada_2015  bajada   455.0       3.154        8.954      0.402       0.392
         108          3  temporada_2016   valle   780.0       0.613        1.330      1.026       0.808
         109          3  temporada_2016  subida  1235.0       2.171        7.212      1.302       1.438
         110          3  temporada_2016    pico   780.0       3.858        9.433      1.219       1.096
         111          3  temporada_2016  bajada   585.0       1.131        2.820      0.633       0.530
         132          3  temporada_2022   valle  1625.0       

In [12]:
rd = {k: p2.readings(v, inp["serie"]) for k, v in {"L": L, **variants}.items()}
pd.concat({k: v["hu0007_4"] for k, v in rd.items()}, names=["variante"])


horizonte          bloque  precision_050  recall_050    f1_050  razon_mae  razon_rmse  cumple_precision_050  cumple_recall_050  cumple_f1_050  cumple_razon_mae  cumple_razon_rmse  cumple_clasificacion  \
variante                                                                                                                                                                                                                
L        0           2  temporada_2010       0.745455    0.700855  0.722467   0.938085    0.919739                  True              False           True             False              False                 False   
         1           2  temporada_2015       0.865169    0.694794  0.772455   0.945156    1.053429                  True              False           True             False              False                 False   
         2           2  temporada_2016       0.609071    0.713924  0.657343   1.100044    1.071417                  True              False          False             False              False                 False   
         3           2  temporada_2017       0.663921    0.743970  0.703316   0.800489    0.875387                  True              False           True              True              False                 False   
         4           2  temporada_2022       0.728155    0.696056  0.711905   0.961816    1.010115                  True              False           True             False              False                 False   
         5           2  temporada_2023       0.888545    0.827719  0.857428   0.846863    0.940025                  True               True           True              True              False                  True   
         6           2  temporada_2024       0.752058    0.852975  0.799344   1.259388    1.565843                  True               True           True             False              False                  True   
         7           3  temporada_2010       0.731844    0.551282  0.630807   0.896763    0.890730                  True              False          False             False              False                 False   
         8           3  temporada_2015       0.875000    0.515260  0.648588   0.894666    1.044787                  True              False          False             False              False                 False   
         9           3  temporada_2016       0.599078    0.658228  0.627262   1.130717    1.142470                 False              False          False             False              False                 False   
         10          3  temporada_2017       0.652095    0.664193  0.658065   0.754253    0.840480                  True              False          False              True               True                 False   
         11          3  temporada_2022       0.727723    0.682135  0.704327   0.959157    1.063237                  True              False           True             False              False                 False   
         12          3  temporada_2023       0.880342    0.794033  0.835020   0.790613    0.887695                  True              False           True              True              False                 False   
         13          3  temporada_2024       0.725221    0.863477  0.787814   1.382340    1.777912                  True               True           True             False              False                  True   
         14          4  temporada_2010       0.714286    0.358974  0.475921   0.875766    0.858832                  True              False          False             False              False                 False   
         15          4  temporada_2015       0.900990    0.326750  0.479578   0.846978    0.993067                  True              False          False              True              False                 False   
         16          4  temporada_2016       0.608108    0.572152  0.590377   1.150254    1.175559         

In [13]:
# Lectura de Rosa: la probabilidad NB2 desde la μ mezclada (5a con la misma μ), luego Platt
same = p5.same_mu_reading(saved, inp["serie"])
print("Reproducción de L con NB2 (diferencia máxima):", same["diferencia_max_reproduccion_L"])
same_cmp = p5.compare({"evaluaciones": same["evaluaciones"]}, L)
display(same_cmp["veredicto"])
same_cmp["metricas"][["horizonte", "bloque", "positivos_cand", "f1_050_cand", "f1_050_ref", "auprc_cand", "auprc_ref",
                      "brier_cand", "brier_ref"]].round(3)


Reproducción de L con NB2 (diferencia máxima): 0.0


,horizonte,veredicto,vetos,vetos_contra_referencia,a_favor_razon_mae,a_favor_razon_rmse,a_favor_f1_050,a_favor_auprc,a_favor_brier,en_contra_razon_mae,en_contra_razon_rmse,en_contra_f1_050,en_contra_auprc,en_contra_brier,a_favor_antelacion,en_contra_antelacion
0,2,compensa (empate),brier,razon_rmse,5,4,1,2,2,0,0,0,2,3,0,1
1,3,compensa (empate),auprc,razon_rmse,4,4,0,1,1,0,0,0,3,2,0,1
2,4,compensa (empate),auprc,razon_rmse,4,4,2,1,2,0,0,0,3,2,0,1


,horizonte,bloque,positivos_cand,f1_050_cand,f1_050_ref,auprc_cand,auprc_ref,brier_cand,brier_ref
3,2,temporada_2010,234.0,0.713,0.722,0.783,0.780,0.030,0.030
8,2,temporada_2015,557.0,0.779,0.772,0.882,0.885,0.051,0.051
9,2,temporada_2016,395.0,0.658,0.657,0.760,0.757,0.059,0.059
15,2,temporada_2022,431.0,0.708,0.712,0.776,0.781,0.055,0.054
16,2,temporada_2023,1039.0,0.862,0.857,0.947,0.947,0.063,0.062
17,2,temporada_2024,857.0,0.800,0.799,0.893,0.893,0.074,0.074
21,3,temporada_2010,234.0,0.631,0.631,0.706,0.706,0.036,0.036
26,3,temporada_2015,557.0,0.656,0.649,0.837,0.839,0.064,0.064
27,3,temporada_2016,395.0,0.631,0.627,0.716,0.709,0.064,0.065
33,3,temporada_2022,431.0,0.705,0.704,0.767,0.771,0.058,0.056


## 5 · Salidas: métricas y MLflow


In [14]:
to_json = lambda d: {k: diag.to_json(v) for k, v in d.items() if isinstance(v, pd.DataFrame)}
content = {
    "dispositivo": device, "guardas": {"G11": g11, "G12": g12, "G13": g13},
    "veredictos": {v: to_json(d) for v, d in dec.items()}, "adoptadas": adopted, "cruces": crosses, "final": final,
    "w_5a": diag.to_json(w_summary), "max_delta_step_5b": diag.to_json(B["ajustes"].drop(columns=["corte"])),
    "conteo_por_fase": diag.to_json(phases.reset_index(level=0)),
    "hu0007_4": {k: diag.to_json(v["hu0007_4"]) for k, v in rd.items()},
    "misma_mu": {"diferencia_max_reproduccion_L": same["diferencia_max_reproduccion_L"], **to_json(same_cmp)},
}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
tracking.log_study(
    "mejora_modelo_fase5_conteo", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "5", "preregistro": "6d2164c", "final": final,
                       **{f"veredicto.{v}": "; ".join(f"h{r.horizonte}: {r.veredicto}" for r in d["veredicto"].itertuples())
                          for v, d in dec.items()},
                       **{f"xgb.{k}": str(x) for k, x in device.items()}},
            "metricas": {}},
    artifacts=[METRICS], tables={f"veredicto_{v}": d["veredicto"] for v, d in dec.items()},
    description="Fase 5 del estudio 06: mezcla con la persistencia, max_delta_step y peso por temporada sobre L.")


C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio 3259773a81f14abeb94cbcbc2366647f en 'denguekay/mejora_modelo_fase5_conteo' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'3259773a81f14abeb94cbcbc2366647f'